In [1]:
from openai import OpenAI
import os
import json
import hashlib
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def load_json(path):
    return json.load(open(path , 'r', encoding='utf-8'))

c:\Users\ASUS\Desktop\crawl_fb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

In [3]:
messages = [{"role": "user", 
             "content": "Who are you"}]

completion = client.chat.completions.create(
    model="qwen3-8b", 
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)

In [4]:
is_answering = False  # Indicates whether the response phase has started
print("\n" + "=" * 20 + "Thinking process" + "=" * 20)
for chunk in completion:
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "Full response" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)


====================Thinking process====================
Okay, the user asked "Who are you?" I need to respond in a friendly and informative way. Let me start by introducing myself as Qwen, the large language model developed by Alibaba Cloud. I should mention my capabilities, like answering questions, creating content, and engaging in conversations. It's important to highlight that I can assist with various tasks, from solving problems to providing information. I should keep the tone welcoming and invite the user to ask questions or share needs. Let me make sure the response is clear and concise, avoiding any technical jargon. Also, I should check for any recent updates or features that I should mention. Alright, that should cover it.
====================Full response====================
Hello! I am Qwen, a large language model developed by Alibaba Cloud. I was designed to assist with a wide range of tasks, such as answering questions, creating content, engaging in conversations, and 

In [ ]:
import re

def build_vietnamese_normalization_prompt(comments):
    """
    Build a prompt to normalize Vietnamese comments while preserving emoji/emoticons.

    Args:
        comments: str or list[str]

    Returns:
        str: prompt text ready to send to the model
    """
    if isinstance(comments, str):
        comments = [comments]

    prompt = f"""
Bạn là bộ chuẩn hóa văn bản tiếng Việt.
Nhiệm vụ: chuẩn hóa chính tả, dấu, viết tắt, teencode về tiếng Việt chuẩn.

Input comments:
{json.dumps(comments, ensure_ascii=False, indent=2)}

Chỉ trả về JSON array, không thêm markdown/code block.
""".strip()
    return prompt


def _extract_json_array(text):
    """Extract the first JSON array from model output."""
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n|\n```$", "", text.strip())

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
    except Exception:
        pass

    match = re.search(r"\[[\s\S]*\]", text)
    if not match:
        raise ValueError("Model output does not contain a JSON array.")

    data = json.loads(match.group(0))
    if not isinstance(data, list):
        raise ValueError("Parsed JSON is not an array.")
    return data


def normalize_vietnamese_comments(client, comments, model="qwen3-max"):
    """
    Normalize Vietnamese comments using LLM.

    Returns:
        list[str]
    """
    single_input = isinstance(comments, str)
    comments_list = [comments] if single_input else list(comments)

    prompt = build_vietnamese_normalization_prompt(comments_list)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Bạn là trợ lý chuẩn hóa văn bản tiếng Việt. Tuân thủ output JSON chính xác."
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    raw_text = response.choices[0].message.content or ""
    normalized = _extract_json_array(raw_text)

    if len(normalized) != len(comments_list):
        raise ValueError(
            f"Output length mismatch: expected {len(comments_list)}, got {len(normalized)}"
        )

    if single_input:
        return normalized[0]
    return normalized

In [6]:

comments = ["tr oi dep qa :)))", "nay zui vler 😂😂"]
normalized = normalize_vietnamese_comments(client, comments)
print(normalized)

['trời ơi đẹp quá :)))', 'này vui vler 😂😂']
